'''''# De Bruijn graphs

You will be implementing one of the primary assembly algorithms from short-read data that is used today. We will implement a simple form of the algorithm where we _assume perfect sequencing_. That is, everything is sequenced exactly once and there are no errors or variants in the sequencing. 

A graph is composed of **nodes** and **edges** and we will need to develop a data strcture to track edges between nodes in our graph. We have provided the basic class structure as well as descriptions of functions to `add_edge` and `remove_edge` from the graph. You will need to implement these functions in order to then build the de Bruijn graph. 

In our implementation below, we use a `defaultdict` data structure to hold a list of all edges in the graph where all "right" nodes connected to a "left" node are stored in a list for that node.

```
build_debruijn_graph:
define substring length k and input string
For each k-length substring of input:
  split k mer into left and right k-1 mer
  add k-1 mers as nodes with a directed edge from left k-1 mer to right k-1 mer
```

---
## Eulerian walk

To continue our implementation from last class, we will use our De Bruijn graph to output a valid sequence from the assembly. This is implemented as a recursive algorithm by considering all valid edges. You will notice that as you change $k$, we are able to better recapitulate our sequence depending on how repetitive it is. In a more complex implementation of a Eulerian walk there are heuristics and defined rules for determining the validity of traversing a specific edge in the graph to result in a full graph-traversal. One of these methods is to traverse the graph in a depth first manner to avoid sectioning off any part of the graph in the traversal. In our implementation we will ignore these for simplicity.

```
eulerian_walk:
Beginning at first_node as node

For node:
    follow a random valid edge from node
    remove edge
    recurse
```


In [3]:
import gzip


def read_fastq(filename):
    """Read sequences from a FASTQ file.

    Args:
        filename (str): Path to FASTQ file.

    Returns:
        list: List of DNA sequence strings (quality scores ignored).

    Example:
        >>> reads = read_fastq("test_reads.fastq")
        >>> len(reads)
        10
    """
    opener = gzip.open if filename.endswith(".gz") else open
    sequences = []
    with opener(filename, "rt") as fasta:
        for line_count, line in enumerate(fasta, 1):
            if line_count % 4 == 2:
                sequences.append(line.strip().upper())
    return sequences

In [1]:
"""De Bruijn Graph Genome Assembly Module.

This module provides classes and functions for constructing De Bruijn graphs
from sequencing reads and performing genome assembly using Eulerian path
traversal.
"""
from collections import defaultdict
import random
import sys
sys.setrecursionlimit(10_000_000)


class DeBruijnGraph:
    """Main class for De Bruijn graphs and genome assembly.

    This class builds De Bruijn graphs from sequencing reads and performs
    genome assembly by finding Eulerian paths through connected components
    of the graph.

    Attributes:
        graph (defaultdict): Adjacency list representation of graph edges.
            Keys are (k-1)-mers, values are lists of adjacent (k-1)-mers.
        k (int): The k-mer size used for graph construction.

    Example:
        >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
        >>> dbg = DeBruijnGraph(reads, k=6)
        >>> contigs = dbg.assemble_contigs(seed=42)
        >>> len(contigs) > 0
        True
    """

    def __init__(self, reads, k):
        """Initialize De Bruijn graph from sequencing reads.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer size for graph construction.

        Example:
            >>> reads = ["ATGGCG", "GCGTGC", "TGCAAC"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> len(dbg.graph) > 0
            True
        """
        self.graph = defaultdict(list)
        self.k = k
        self.build_graph_from_reads(reads, k)

    def add_edge(self, left, right):
        """Add a directed edge to the graph.

        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> "TGG" in dbg.graph["ATG"]
            True

        Pseudocode:
        append right to graph[left]
        """
        self.graph[left].append(right)

    def remove_edge(self, left, right):
        """Remove a directed edge from the graph.

        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> dbg.remove_edge("ATG", "TGG")
            >>> len(dbg.graph["ATG"])
            0

        Pseudocode:
        remove one occurrence of right from graph[left]

        """
        self.graph[left].remove(right)

    def build_graph_from_reads(self, reads, k):
        """Build De Bruijn graph from multiple sequencing reads.

        Extracts all k-mers from all reads and adds edges between
        consecutive (k-1)-mers within each k-mer.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer length for graph construction.

        Example:
            >>> reads = ["ATGGC", "TGGCA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.build_graph_from_reads(reads, 4)
            >>> "ATG" in dbg.graph
            True

        Pseudocode:
        for each read in reads:
            for i from 0 to len(read) − k:
                kmer  = read[i : i+k]
                left  = kmer[0 : k−1] # drop last base for prefix
                right = kmer[1 : k]   # drop first base for suffix
                add_edge(left, right)


        """
        for read_num, read in enumerate(reads, 1):
            for i in range(len(read) - k + 1):
                kmer = read[i:i + k]
                left = kmer[:k - 1]
                right = kmer[1:]

                # If duplicate seen already in the built graph, skip over
                if right in self.graph[left]:
                    continue

                self.add_edge(left, right)
                #print(f"{kmer}: {left} -> {right}")  # print for toy

            if read_num % 250_000 == 0:
                print(f"...processed {read_num:,} reads")  # print for yeast data 

    def eulerian_walk(self, node, graph, seed=None, tour=None):
        """Perform recursive Eulerian walk on a graph component.

        This is a recursive function that follows all edges from a node
        to traverse the graph, building a path in reverse order.

        Args:
            node (str): Current node to traverse from.
            graph (defaultdict): Graph or subgraph to traverse.
            seed (int, optional): Seed for random edge selection.

        Returns:
            list: List of (k-1)-mers traversed (in reverse order).

        Example:
            >>> reads = ["ATGGCG"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> graph_copy = defaultdict(list, dbg.graph)
            >>> tour = dbg.eulerian_walk("ATG", graph_copy, seed=42)
            >>> len(tour) > 0
            True

        Pseudocode:
        tour = []
        while graph[node] still has edges:
            next_node = pick a random neighbor from graph[node]
            remove_edge(node, next_node) #consume edge                 
            tour.extend(eulerian_walk(next_node, graph)) #recurse 
        tour.append(node) # append on the way back out
        return tour # deepest node was added first, want to reverse it at the end


        """
        # Prevent infinite looping problem 
        if seed is not None:
            random.seed(seed)
        if tour is None:
            tour = []

        while graph[node]:
            next_node = random.choice(graph[node])
            graph[node].remove(next_node)
            #print(f"walk: {node} -> {next_node}")                # print for toy data
            self.eulerian_walk(next_node, graph, tour=tour)

        tour.append(node)
        return tour
        



    def assemble_contigs(self, seed=None):
        """Assemble all contigs from the De Bruijn graph.

        Finds all connected components and generates an Eulerian path
        for each component, producing multiple assembled contigs.

        Args:
            seed (int, optional): Random seed for reproducible assembly.

        Returns:
            list: List of assembled contig sequences (DNA strings).

        Example:
            >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
            >>> dbg = DeBruijnGraph(reads, k=6)
            >>> contigs = dbg.assemble_contigs(seed=42)
            >>> all(isinstance(c, str) for c in contigs)
            True

        Pseudocode:
        seed the RNG once
        graph = a copy of self.graph
        compute indegree of every node

        contigs = []
        #Start from ideal nodes which have more outgoing than incoming edges to start path
        for node in nodes where out > in:
            while graph[node] has edges:
                tour = reverse(eulerian_walk(node, graph))
                contigs.append(tour_to_sequence(tour))
        #Anything left over is a cycle, start anywhere that still has edges
        for node in all nodes:
            while graph[node] has edges:
                tour = reverse(eulerian_walk(node, graph))
                contigs.append(tour_to_sequence(tour))
        return contigs


        """
        
        if seed is not None:
            random.seed(seed)  
 
        graph = defaultdict(list, {node: list(neighbors) for node, neighbors in self.graph.items()})
 
        
        in_degree = defaultdict(int)
        for neighbors in graph.values():
            for neighbor in neighbors:
                in_degree[neighbor] += 1
 
        all_nodes = list(graph.keys())
        #Start from ideal nodes which have more outgoing than incoming edges to start path
        start_nodes = [node for node in all_nodes if len(graph[node]) > in_degree[node]]
 
        contigs = []
 
        for node in start_nodes:
            while graph[node]:
                tour = self.eulerian_walk(node, graph)
                tour.reverse()  
                contig = self.tour_to_sequence(tour)
                contigs.append(contig)
                print(f"contig {len(contigs)}: {len(contig):,} bp") #print
 
        #Anything left over is a cycle, start anywhere that still has edges
        for node in all_nodes:
            while graph[node]:
                tour = self.eulerian_walk(node, graph)
                tour.reverse()  
                contig = self.tour_to_sequence(tour)
                contigs.append(contig)
                print(f"contig {len(contigs)}: {len(contig):,} bp") #print
 
        return contigs

    def tour_to_sequence(self, tour):
        """Convert a tour of (k-1)-mers into a DNA sequence.

        Args:
            tour (list): List of (k-1)-mer strings in order.

        Returns:
            str: Assembled DNA sequence.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> tour = ['ATG', 'TGG', 'GGC', 'GCG']
            >>> dbg.tour_to_sequence(tour)
            'ATGGCG'

        Pseudocode:

        if tour is empty: return ""
        seq = [tour[0]]
        for node in tour[1:]:
            seq.append(node[-1])
        return "".join(seq)


        """
        if not tour:
            return ""
        seq = [tour[0]]
        for node in tour[1:]:
            seq.append(node[-1])

        return "".join(seq)

    def get_assembly_stats(self, contigs):
        """Calculate assembly statistics for assembled contigs.

        Args:
            contigs (list): List of contig sequences.

        Returns:
            dict: Dictionary containing assembly statistics:
                - num_contigs: Total number of contigs
                - total_length: Total assembled sequence length
                - longest_contig: Length of longest contig
                - shortest_contig: Length of shortest contig
                - mean_length: Mean contig length
                - n50: N50 statistic

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA", "CCCCCCCCCC"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> stats = dbg.get_assembly_stats(contigs)
            >>> stats['num_contigs']
            3

        Pseudocode:

        lengths = sorted(len(c) for c in contigs, descending)
        num_contigs, total_length, longest = lengths[0], shortest = lengths[-1], mean
        n50: walk down the sorted lengths keeping a running sum;
            the first length where running sum >= total_length / 2 is the N50
        (handle empty contig case by returning zeros)

        """
        if not contigs:
            return {"num_contigs": 0, "total_length": 0, "longest_contig": 0,
                    "shortest_contig": 0, "mean_length": 0.0, "n50": 0}
 
        lengths = sorted((len(c) for c in contigs), reverse=True)
        total_length = sum(lengths)
 
        n50 = 0
        running_sum = 0
        for length in lengths:
            running_sum += length
            if running_sum >= total_length / 2:
                n50 = length
                break
 
        return {
            "num_contigs": len(lengths),
            "total_length": total_length,
            "longest_contig": lengths[0],
            "shortest_contig": lengths[-1],
            "mean_length": total_length / len(lengths),
            "n50": n50,
        }

    def write_fasta(self, contigs, filename):
        """Write assembled contigs to a FASTA file.

        Args:
            contigs (list): List of contig sequences.
            filename (str): Output FASTA filename.

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.write_fasta(contigs, "output.fasta")

        Pseudocode:

        for i, contig in enumerate(contigs, 1):
        write ">contig_i length=N"
        write the sequence 


        """
        with open(filename, "w") as f:
            for i, contig in enumerate(contigs, 1):
                f.write(f">contig_{i} length={len(contig)}\n")
                f.write(contig + "\n")


---
# Toy Example

In [8]:
print("="*60)
print("EXAMPLE 1: Assembling 9 overlapping reads into one contig")
print("="*60)

toy_reads_1 = [
    "ATGGCGTACG",  # Read 1
    "GGCGTACGTT",  # Read 2: overlaps with Read 1
    "CGTACGTTAC",  # Read 3: overlaps with Read 2
    "TACGTTACCA",  # Read 4: overlaps with Read 3
    "CGTTACCATG",  # Read 5: overlaps with Read 4
    "TTACCATGGG",  # Read 6: overlaps with Read 5
    "ACCATGGGCC",  # Read 7: overlaps with Read 6
    "CATGGGCCTA",  # Read 8: overlaps with Read 7
    "TGGGCCTAAA"   # Read 9: overlaps with Read 8
]

print(f"\nInput: {len(toy_reads_1)} reads")
print("First read:  ", toy_reads_1[0])
print("Last read:   ", toy_reads_1[-1])
print(f"\nBuilding De Bruijn graph with k=6...")

dbg = DeBruijnGraph(toy_reads_1, k=6)
contigs = dbg.assemble_contigs(seed=42)

print(f"\nNumber of contigs: {len(contigs)}")
for contig in contigs:
    print(contig)

EXAMPLE 1: Assembling 9 overlapping reads into one contig

Input: 9 reads
First read:   ATGGCGTACG
Last read:    TGGGCCTAAA

Building De Bruijn graph with k=6...
ATGGCG: ATGGC -> TGGCG
TGGCGT: TGGCG -> GGCGT
GGCGTA: GGCGT -> GCGTA
GCGTAC: GCGTA -> CGTAC
CGTACG: CGTAC -> GTACG
GTACGT: GTACG -> TACGT
TACGTT: TACGT -> ACGTT
ACGTTA: ACGTT -> CGTTA
CGTTAC: CGTTA -> GTTAC
GTTACC: GTTAC -> TTACC
TTACCA: TTACC -> TACCA
TACCAT: TACCA -> ACCAT
ACCATG: ACCAT -> CCATG
CCATGG: CCATG -> CATGG
CATGGG: CATGG -> ATGGG
ATGGGC: ATGGG -> TGGGC
TGGGCC: TGGGC -> GGGCC
GGGCCT: GGGCC -> GGCCT
GGCCTA: GGCCT -> GCCTA
GCCTAA: GCCTA -> CCTAA
CCTAAA: CCTAA -> CTAAA
walk: ATGGC -> TGGCG
walk: TGGCG -> GGCGT
walk: GGCGT -> GCGTA
walk: GCGTA -> CGTAC
walk: CGTAC -> GTACG
walk: GTACG -> TACGT
walk: TACGT -> ACGTT
walk: ACGTT -> CGTTA
walk: CGTTA -> GTTAC
walk: GTTAC -> TTACC
walk: TTACC -> TACCA
walk: TACCA -> ACCAT
walk: ACCAT -> CCATG
walk: CCATG -> CATGG
walk: CATGG -> ATGGG
walk: ATGGG -> TGGGC
walk: TGGGC -> GGGC

---
# Real data
This section utilizes a FASTQ file with 2.4 million "perfect" 150 bp reads simulated
from the _S. cerevisiae_ (yeast) genome (`R64`) and tasks your program to ingest, assemble, and traverse
the genome with statistical output report.

In [ ]:
"""Driver program for genome assembly using De Bruijn graphs.

This cell runs the completed DeBruijnGraph class on a FASTQ file of perfect
(error-free, variant-free) single-end reads simulated from a reference genome,
then reports assembly statistics.

Usage:
    Run all cells above this one first, then set the configuration values
    below and run this cell.

Expected input:
    - FASTQ or gzipped FASTQ (.fq, .fastq, .fq.gz, .fastq.gz)
    - Perfect single-end reads simulated with wgsim

Output:
    - <output_fasta>: Assembled contigs
    - <stats_file>: Detailed statistics report
"""

import time
from datetime import datetime

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
INPUT_FILE = "/Users/juliannemurthy/Downloads/project04/data/sacCer_R64_150SE_30x_wgsim.fq.gz"
OUTPUT_FASTA = "/Users/juliannemurthy/Downloads/project04/genome_assembly.fasta"
STATS_FILE = "/Users/juliannemurthy/Downloads/project04/genome_assembly_stats.txt"
K_MER_SIZE = 31
# Size (bp) of the reference the reads were simulated from; used for the
# coverage and assembly-fraction estimates. Set to None if unknown.
GENOME_SIZE = 12_071_326


def _pct(part, whole):
    """Return part/whole as a percentage, or 0.0 when whole is zero."""
    return (part / whole * 100) if whole else 0.0


def write_statistics_file(
    stats_file,
    input_file,
    num_reads,
    avg_read_length,
    k_mer_size,
    num_nodes,
    num_edges,
    stats,
    contig_lengths,
    timing,
    genome_size,
    coverage_estimate,
    assembly_fraction,
):
    """Write comprehensive assembly statistics to a text file.

    Args:
        stats_file (str): Output filename.
        input_file (str): Input FASTQ filename.
        num_reads (int): Number of reads processed.
        avg_read_length (float): Average read length.
        k_mer_size (int): K-mer size used.
        num_nodes (int): Number of graph nodes.
        num_edges (int): Number of distinct graph edges.
        stats (dict): Assembly statistics.
        contig_lengths (list): Contig lengths sorted longest first.
        timing (dict): Timing information, including 'total_time'.
        genome_size (int or None): Reference genome size in bp.
        coverage_estimate (float or None): Estimated sequencing coverage.
        assembly_fraction (float or None): Assembly size as % of genome.
    """
    rule = "-" * 80 + "\n"
    banner = "=" * 80 + "\n"

    with open(stats_file, "w") as f:
        f.write(banner)
        f.write("GENOME ASSEMBLY STATISTICS\n")
        f.write(banner + "\n")

        f.write(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Input file:    {input_file}\n")
        f.write(f"K-mer size:    {k_mer_size}\n\n")

        f.write(rule + "INPUT DATA\n" + rule)
        f.write(f"Number of reads:         {num_reads:,}\n")
        f.write(f"Average read length:     {avg_read_length:.1f} bp\n")
        f.write(f"Total sequencing data:   {num_reads * avg_read_length:,.0f} bp\n")
        if genome_size:
            f.write(f"Reference genome size:   {genome_size:,} bp\n")
            f.write(f"Estimated coverage:      {coverage_estimate:.1f}x\n")
        f.write(f"Read time:               {timing['read_time']:.2f} seconds\n\n")

        f.write(rule + "DE BRUIJN GRAPH CONSTRUCTION\n" + rule)
        f.write(f"Graph nodes:             {num_nodes:,}\n")
        f.write(f"Graph edges:             {num_edges:,}\n")
        avg_degree = num_edges / num_nodes if num_nodes else 0.0
        f.write(f"Average out-degree:      {avg_degree:.2f}\n")
        f.write(f"Construction time:       {timing['graph_time']:.2f} seconds\n\n")

        f.write(rule + "ASSEMBLY RESULTS\n" + rule)
        f.write(f"Number of contigs:       {stats['num_contigs']:,}\n")
        f.write(f"Total assembly length:   {stats['total_length']:,} bp\n")
        if genome_size:
            f.write(f"Assembly vs. genome:     {assembly_fraction:.2f}%\n")
        f.write(f"Longest contig:          {stats['longest_contig']:,} bp\n")
        f.write(f"Shortest contig:         {stats['shortest_contig']:,} bp\n")
        f.write(f"Mean contig length:      {stats['mean_length']:,.1f} bp\n")
        f.write(f"N50:                     {stats['n50']:,} bp\n")
        f.write(f"Assembly time:           {timing['assembly_time']:.2f} seconds\n\n")

        f.write(rule + "TOP 20 LONGEST CONTIGS\n" + rule)
        for i, length in enumerate(contig_lengths[:20], 1):
            f.write(f"{i:3d}. {length:10,} bp\n")
        f.write("\n")

        f.write(rule + "CONTIG LENGTH DISTRIBUTION\n" + rule)
        bins = [
            ("all", len(contig_lengths)),
            (">100kb", sum(1 for x in contig_lengths if x > 100_000)),
            (">50kb", sum(1 for x in contig_lengths if x > 50_000)),
            (">10kb", sum(1 for x in contig_lengths if x > 10_000)),
            (">5kb", sum(1 for x in contig_lengths if x > 5_000)),
            (">1kb", sum(1 for x in contig_lengths if x > 1_000)),
            (">500bp", sum(1 for x in contig_lengths if x > 500)),
        ]
        for bin_name, count in bins:
            f.write(f"Contigs {bin_name:8s}:     {count:,}\n")
        f.write("\n")

        total_time = timing["total_time"]
        f.write(rule + "TIMING SUMMARY (excluding file output)\n" + rule)
        for label, key in (
            ("Read time:", "read_time"),
            ("Graph construction:", "graph_time"),
            ("Assembly:", "assembly_time"),
        ):
            f.write(f"{label:25s}{timing[key]:8.2f} seconds "
                    f"({_pct(timing[key], total_time):5.1f}%)\n")
        f.write(f"{'Total time:':25s}{total_time:8.2f} seconds "
                f"({total_time / 60:.2f} minutes)\n\n")

        f.write(banner + "END OF REPORT\n" + banner)


def count_graph(graph):
    """Count nodes and distinct edges in an adjacency-list graph.

    Nodes that only receive edges may not be keys of the adjacency dict, so
    they are counted separately. Duplicate neighbor entries (the same k-mer
    seen in several reads) are counted once.

    Args:
        graph (dict): Mapping of (k-1)-mer -> iterable of neighbor (k-1)-mers.

    Returns:
        tuple: (num_nodes, num_edges)
    """
    num_edges = 0
    sink_only = set()
    for neighbors in graph.values():
        distinct = set(neighbors)
        num_edges += len(distinct)
        for node in distinct:
            if node not in graph:  # membership test does not insert keys
                sink_only.add(node)
    return len(graph) + len(sink_only), num_edges


def assemble_genome(
    input_file=INPUT_FILE,
    output_fasta=OUTPUT_FASTA,
    stats_file=STATS_FILE,
    k_mer_size=K_MER_SIZE,
    genome_size=GENOME_SIZE,
):
    """Run the complete assembly pipeline.

    1. Load FASTQ reads
    2. Build the De Bruijn graph
    3. Assemble contigs
    4. Calculate statistics
    5. Write output files

    Returns:
        dict: Assembly results with keys 'dbg', 'contigs', 'stats', 'timing',
            'graph_stats', 'coverage', 'output_fasta', and 'stats_file'.
    """
    print("=" * 80)
    print("GENOME ASSEMBLY PIPELINE")
    print("=" * 80)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    timing = {}

    # ------------------------------------------------------------------ 1
    print("STEP 1: Loading sequencing reads")
    print("-" * 80)
    print(f"Input file: {input_file}\n")

    start_time = time.time()
    reads = read_fastq(input_file)
    timing["read_time"] = time.time() - start_time

    num_reads = len(reads)
    if num_reads == 0:
        raise ValueError(f"No reads found in {input_file}")
    total_bases = sum(len(read) for read in reads)
    avg_read_length = total_bases / num_reads

    print(f"Reads loaded: {num_reads:,}")
    print(f"Total bases: {total_bases:,} bp")
    print(f"Average read length: {avg_read_length:.1f} bp")
    print(f"Time elapsed: {timing['read_time']:.2f} seconds")
    print("\nSample reads:")
    print(f"  First: {reads[0][:80]}...")
    print(f"  Last:  {reads[-1][:80]}...\n")

    # ------------------------------------------------------------------ 2
    print("STEP 2: Building De Bruijn graph")
    print("-" * 80)
    print(f"K-mer size: {k_mer_size}")
    print(f"Building graph from {num_reads:,} reads...\n")

    start_time = time.time()
    dbg = DeBruijnGraph(reads, k=k_mer_size)
    timing["graph_time"] = time.time() - start_time

    del reads  # the graph holds everything needed; free the read list

    num_nodes, num_edges = count_graph(dbg.graph)
    avg_degree = num_edges / num_nodes if num_nodes else 0.0

    print("Graph construction complete!")
    print(f"  Nodes (distinct {k_mer_size - 1}-mers): {num_nodes:,}")
    print(f"  Edges (distinct {k_mer_size}-mers):   {num_edges:,}")
    print(f"  Average out-degree: {avg_degree:.2f}")
    print(f"Time elapsed: {timing['graph_time']:.2f} seconds\n")

    # ------------------------------------------------------------------ 3
    print("STEP 3: Assembling contigs")
    print("-" * 80)
    print("Traversing the graph to build contigs...\n")

    start_time = time.time()
    contigs = dbg.assemble_contigs()
    timing["assembly_time"] = time.time() - start_time
    timing["total_time"] = (
        timing["read_time"] + timing["graph_time"] + timing["assembly_time"]
    )

    print("Assembly complete!")
    print(f"  Contigs generated: {len(contigs):,}")
    print(f"Time elapsed: {timing['assembly_time']:.2f} seconds\n")

    # ------------------------------------------------------------------ 4
    print("STEP 4: Calculating assembly statistics")
    print("-" * 80)

    stats = dbg.get_assembly_stats(contigs)
    contig_lengths = sorted((len(c) for c in contigs), reverse=True)

    print("Assembly statistics:")
    print(f"  Number of contigs:     {stats['num_contigs']:,}")
    print(f"  Total assembly length: {stats['total_length']:,} bp")
    print(f"  Longest contig:        {stats['longest_contig']:,} bp")
    print(f"  Shortest contig:       {stats['shortest_contig']:,} bp")
    print(f"  Mean contig length:    {stats['mean_length']:,.1f} bp")
    print(f"  N50:                   {stats['n50']:,} bp\n")

    print("Top 10 longest contigs:")
    for i, length in enumerate(contig_lengths[:10], 1):
        print(f"  {i:2d}. {length:,} bp")
    print()

    if genome_size:
        coverage_estimate = total_bases / genome_size
        assembly_fraction = _pct(stats["total_length"], genome_size)
        print("Genome coverage analysis:")
        print(f"  Reference genome size:         {genome_size:,} bp")
        print(f"  Estimated sequencing coverage: {coverage_estimate:.1f}x")
        print(f"  Assembly size vs. genome:      {assembly_fraction:.1f}%")
        print("  (Near 200% usually means reverse-complement duplicate "
              "contigs were not removed.)\n")
    else:
        coverage_estimate = assembly_fraction = None

    # ------------------------------------------------------------------ 5
    print("STEP 5: Writing output files")
    print("-" * 80)

    dbg.write_fasta(contigs, output_fasta)
    print(f"✓ Contigs written to: {output_fasta}")

    write_statistics_file(
        stats_file,
        input_file,
        num_reads,
        avg_read_length,
        k_mer_size,
        num_nodes,
        num_edges,
        stats,
        contig_lengths,
        timing,
        genome_size,
        coverage_estimate,
        assembly_fraction,
    )
    print(f"✓ Statistics written to: {stats_file}\n")

    print("=" * 80)
    print("ASSEMBLY COMPLETE")
    print("=" * 80)
    print(f"Total time (excluding file output): {timing['total_time']:.2f} seconds "
          f"({timing['total_time'] / 60:.2f} minutes)")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    return {
        "dbg": dbg,
        "contigs": contigs,
        "stats": stats,
        "timing": timing,
        "graph_stats": {
            "num_nodes": num_nodes,
            "num_edges": num_edges,
            "avg_degree": avg_degree,
        },
        "coverage": coverage_estimate,
        "output_fasta": output_fasta,
        "stats_file": stats_file,
    }


print("\n" + "=" * 80)
print("GENOME DE BRUIJN GRAPH ASSEMBLY")
print("Student Assignment Driver Program")
print("=" * 80 + "\n")

result = assemble_genome()

print("=" * 80)
print("FIVE LONGEST ASSEMBLED CONTIGS")
print("=" * 80 + "\n")

for i, contig in enumerate(sorted(result["contigs"], key=len, reverse=True)[:5], 1):
    preview = contig[:100] + "..." if len(contig) > 100 else contig
    print(f">contig_{i} length={len(contig)}")
    print(preview + "\n")

print("=" * 80)
print("✓ ASSEMBLY COMPLETE")
print("=" * 80)
print("\nResults saved to:")
print(f"  • {result['output_fasta']} - {result['stats']['num_contigs']:,} assembled contigs")
print(f"  • {result['stats_file']} - detailed assembly statistics")
print("\nKey metrics:")
print(f"  • Total assembly: {result['stats']['total_length']:,} bp")
print(f"  • N50: {result['stats']['n50']:,} bp")
print(f"  • Longest contig: {result['stats']['longest_contig']:,} bp")
if result["coverage"] is not None:
    print(f"  • Coverage: {result['coverage']:.1f}x")
print()


GENOME DE BRUIJN GRAPH ASSEMBLY
Student Assignment Driver Program

GENOME ASSEMBLY PIPELINE
Start time: 2026-10-13 18:42:38

STEP 1: Loading sequencing reads
--------------------------------------------------------------------------------
Input file: /Users/juliannemurthy/Downloads/project04/data/sacCer_R64_150SE_30x_wgsim.fq.gz

Reads loaded: 2,414,266
Total bases: 362,139,900 bp
Average read length: 150.0 bp
Time elapsed: 2.64 seconds

Sample reads:
  First: ATCACAAAGGAGACAAAATCGTTGATAAAAAGTGCATCAACATTCCCAGCCAGAAAATGCACATCATAAAGACATGTTAT...
  Last:  ATTTAGCTCGCTTGTTTCTTGATTTTTTAGTCTCAATTTCATATCGTTAATAAAATCCTTGTGAGCTTCTGAGTTTAAAG...

STEP 2: Building De Bruijn graph
--------------------------------------------------------------------------------
K-mer size: 31
Building graph from 2,414,266 reads...

...processed 250,000 reads
...processed 500,000 reads
...processed 750,000 reads
...processed 1,000,000 reads
...processed 1,250,000 reads
...processed 1,500,000 reads
...processed 1,750,